In [ ]:
from google.colab import drive; import os; drive.mount('/content/drive'); os.makedirs('/content/drive/MyDrive/space_robotics', exist_ok=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import numpy as np
from tqdm import tqdm

# ==========================================
# 0. CONFIG
# ==========================================

SCALING_FACTOR = 1e-6  # meters -> kilometers for distance computation


# ---------- Torch-based CDIST (kept for compatibility, not used in fast path) ----------
def torch_cdist_numpy(a: np.ndarray) -> np.ndarray:
    """
    a: numpy array (N, 3)
    return: NxN numpy distance matrix
    """
    a_t = torch.tensor(a, dtype=torch.float32)
    dist = torch.cdist(a_t, a_t, p=2)
    return dist.cpu().numpy()


# ==========================================
# 1. LOAD SAVED SIMULATION DATA
# ==========================================

load_path = "/content/drive/MyDrive/space_robotics/rect_and_dat_pos.pt"
data = torch.load(load_path, map_location="cpu")

# Unscaled positions in METERS
sat_positions_m = data["sat_positions"].float()      # (N_sats, T, 3) in meters
rect_positions_m = data["rect_positions"].float()    # (N_rect, T, 3) in meters

# For distance calculations, we use KM (to keep numbers nicer)
sat_positions_km = sat_positions_m * SCALING_FACTOR  # (N_sats, T, 3)
rect_positions_km = rect_positions_m * SCALING_FACTOR

visibility_mask = data["visibility_mask"].bool()     # (N_sats, N_rect, T)

N_sats, T, _ = sat_positions_m.shape
N_rect = rect_positions_m.shape[0]

print("Loaded data:")
print("  sat_positions_m:", sat_positions_m.shape, "(meters)")
print("  rect_positions_m:", rect_positions_m.shape, "(meters)")
print("  visibility_mask :", visibility_mask.shape)


# ==========================================
# 2. OPTIONAL: OLD GRAPH CONSTRUCTION (NOT USED BY FAST METHOD)
# ==========================================

def build_graph_for_time_step(
    t: int,
    S_km: np.ndarray,
    sat_positions_km: torch.Tensor,
    rect_positions_km: torch.Tensor,
    visibility_mask: torch.Tensor,
    sat_sat_range_km: float = 1000.0,
    source_sat_range_km: float = 1000.0,
    ignore_mask: bool = False,
    ignore_source_sat_range_km: bool = True,
    ignore_sat_sat_range_km: bool = True,
):
    """
    Original graph builder (kept only for reference / future experiments).
    The fast path computation below does NOT call this function.
    """
    import networkx as nx  # local import so it's optional

    G = nx.Graph()

    G.add_node("S")
    for i in range(N_sats):
        G.add_node(f"sat_{i}")
    for r in range(N_rect):
        G.add_node(f"rect_{r}")

    sats_t = sat_positions_km[:, t, :].numpy()   # (N_sats, 3)
    rect_t = rect_positions_km[:, t, :].numpy()  # (N_rect, 3)

    # Source → Satellite edges
    d_S_sat = np.linalg.norm(sats_t - S_km, axis=1)
    for i, d in enumerate(d_S_sat):
        if ignore_source_sat_range_km or (d <= source_sat_range_km):
            G.add_edge("S", f"sat_{i}", weight=float(d))

    # Satellite ↔ Satellite edges
    dist_matrix = torch_cdist_numpy(sats_t)

    if ignore_sat_sat_range_km:
        i_idx, j_idx = np.where(~np.eye(N_sats, dtype=bool))
    else:
        i_idx, j_idx = np.where(dist_matrix <= sat_sat_range_km)

    for i, j in zip(i_idx, j_idx):
        if i != j:
            G.add_edge(f"sat_{i}", f"sat_{j}", weight=float(dist_matrix[i, j]))

    # Satellite → Rectenna edges
    for i in range(N_sats):
        for r in range(N_rect):
            if ignore_mask or visibility_mask[i, r, t]:
                d = np.linalg.norm(sats_t[i] - rect_t[r])
                G.add_edge(f"sat_{i}", f"rect_{r}", weight=float(d))

    return G


# ==========================================
# 3. HELPER: MAP NODE NAME → 3D POINT (METERS)
# ==========================================

def node_to_xyz_m(node: str, t: int, S_m: np.ndarray) -> list:
    """
    Convert node name ("S", "sat_i", "rect_r") at time t to [x,y,z] in METERS.
    (Kept for compatibility; fast method builds paths directly.)
    """
    if node == "S":
        return S_m.tolist()
    elif node.startswith("sat_"):
        i = int(node[4:])
        return sat_positions_m[i, t, :].tolist()
    elif node.startswith("rect_"):
        r = int(node[5:])
        return rect_positions_m[r, t, :].tolist()
    else:
        raise ValueError(f"Unknown node label: {node}")


# ==========================================
# 4. FAST / VECTORIZED: ALL PATHS FOR ALL RECTENNAS & TIME STEPS
# ==========================================

def compute_all_paths_from_source_fast(S_m: np.ndarray) -> list:
    """
    Fast, vectorized computation of shortest paths:

      - We assume:
         * S connects to all satellites (no cutoff).
         * Satellites are fully connected (we don't need sat->sat hops).
         * Rectennas are reachable only via sats with visibility_mask=True.

      - Therefore, shortest path from S to rect r at time t is:
           S -> best_sat -> rect_r
        where best_sat minimizes:
           cost_i = ||S - sat_i|| + ||sat_i - rect_r||

    Returns
    -------
    paths_matrix : list of lists
        Shape: [N_rect][T]
        - paths_matrix[r][t] == "Null"
              if rectenna r is not activated at time t OR no vis sat
        - otherwise:
              [ [source_xyz], [sat_xyz], [rect_xyz] ]
          where all coords are in METERS.
    """
    # Source in km (for distance computation)
    S_km = torch.tensor(S_m * SCALING_FACTOR, dtype=torch.float32)  # (3,)

    # Make sure main tensors are torch on CPU
    sat_km = sat_positions_km  # (N_sats, T, 3)
    rect_km = rect_positions_km
    vis = visibility_mask      # (N_sats, N_rect, T)

    # Initialize result matrix with "Null"
    paths_matrix = [["Null" for _ in range(T)] for _ in range(N_rect)]

    # Pre-allocate buffers if needed (optional optimization)
    # Main loop over time
    for t in tqdm(range(T), desc="Computing all shortest paths (fast)"):
        # Positions at time t in km
        sats_t = sat_km[:, t, :]      # (N_sats, 3)
        rects_t = rect_km[:, t, :]    # (N_rect, 3)
        vis_t = vis[:, :, t]          # (N_sats, N_rect)

        # Distance from S to each satellite at time t: (N_sats,)
        d_S_sat = torch.linalg.norm(sats_t - S_km, dim=1)  # km

        # Loop over rectennas (small N_rect, so OK)
        for r in range(N_rect):
            # Check which satellites see rect r at time t
            visible_sats = vis_t[:, r]  # (N_sats,) bool
            if not bool(visible_sats.any()):
                # No satellite activates this rectenna at this time
                continue

            idx = visible_sats.nonzero(as_tuple=False).squeeze(1)  # (#vis,)

            sats_vis = sats_t[idx, :]             # (#vis, 3)
            rect_r = rects_t[r, :].unsqueeze(0)   # (1, 3)

            # Distance sat_i -> rect_r in km
            d_sat_rect = torch.linalg.norm(sats_vis - rect_r, dim=1)  # (#vis,)

            # Total cost in km
            total_cost = d_S_sat[idx] + d_sat_rect  # (#vis,)

            # Best satellite index for this rectenna & time
            best_local = torch.argmin(total_cost)
            sat_idx = idx[best_local].item()

            # Build path in METERS: [S, sat_best, rect_r]
            path_xyz = [
                S_m.tolist(),
                sat_positions_m[sat_idx, t, :].tolist(),
                rect_positions_m[r, t, :].tolist(),
            ]
            paths_matrix[r][t] = path_xyz

    return paths_matrix


# ==========================================
# 5. EXAMPLE USAGE
# ==========================================

# Source in METERS (e.g. 7000 km above origin)
S_m = np.array([42_371_000.0, 0.0, 0.0], dtype=float)

paths_matrix = compute_all_paths_from_source_fast(S_m)

print("\nPaths matrix created.")
print(f"Matrix shape: N_rect={len(paths_matrix)}, T={len(paths_matrix[0])}")
print("Example: rectenna 0 at time 0 =>", paths_matrix[0][0])


Loaded data:
  sat_positions_m: torch.Size([1183, 864, 3]) (meters)
  rect_positions_m: torch.Size([50, 864, 3]) (meters)
  visibility_mask : torch.Size([1183, 50, 864])


Computing all shortest paths (fast): 100%|██████████| 864/864 [00:01<00:00, 464.03it/s]


Paths matrix created.
Matrix shape: N_rect=50, T=864
Example: rectenna 0 at time 0 => Null


In [ ]:
import numpy as np
import torch
from collections import deque
from tqdm import tqdm

# ======================================================
# CONSTANTS
# ======================================================

EARTH_RADIUS_KM = 6371.0
EARTH_RADIUS_M  = EARTH_RADIUS_KM * 1000.0  # 6,371,000 m


# ======================================================
# GPU VECTORIZED SEGMENT–SPHERE INTERSECTION
# ======================================================

def segments_intersects_sphere_torch(p1, p2, radius_m, eps=1e-6):
    """
    Vectorized segment–sphere intersection on TORCH (CPU or CUDA).

    Parameters
    ----------
    p1, p2 : torch.Tensor
        Shape (..., 3). Endpoints of segments in METERS, on the same device.
    radius_m : float
        Sphere radius in meters.
    eps : float
        Small tolerance for degenerate segments.

    Returns
    -------
    intersects : torch.BoolTensor
        Shape (...,). True where the segment intersects or touches
        the sphere of radius `radius_m` centered at the origin.
    """
    # Ensure float32 tensors
    p1 = p1.float()
    p2 = p2.float()

    d = p2 - p1          # (..., 3)
    f = p1               # (..., 3)

    a = (d * d).sum(dim=-1)               # (...)
    b = 2.0 * (f * d).sum(dim=-1)         # (...)
    c = (f * f).sum(dim=-1) - (radius_m ** 2)  # (...)

    # Degenerate segments: treat as a point
    deg_mask = a.abs() < eps
    intersects_deg = (f * f).sum(dim=-1) <= (radius_m ** 2)

    # Normal segments
    disc = b * b - 4.0 * a * c
    no_real = disc < 0.0  # no real intersection with infinite line

    sqrt_disc = torch.zeros_like(disc)
    valid_disc_mask = ~no_real
    sqrt_disc[valid_disc_mask] = torch.sqrt(torch.clamp(disc[valid_disc_mask], min=0.0))

    # Avoid division by zero by masking a>eps
    a_safe = torch.where(a.abs() < eps, torch.ones_like(a), a)

    t1 = (-b - sqrt_disc) / (2.0 * a_safe)
    t2 = (-b + sqrt_disc) / (2.0 * a_safe)

    # Intersection with *segment* if t in [0,1]
    intersects_normal = (
        ((t1 >= 0.0) & (t1 <= 1.0)) |
        ((t2 >= 0.0) & (t2 <= 1.0))
    )
    # Must also have real intersection
    intersects_normal = intersects_normal & valid_disc_mask

    # Combine degenerate & normal
    intersects = torch.where(deg_mask, intersects_deg, intersects_normal)
    return intersects


# ======================================================
# MAIN CUDA-ACCELERATED MULTIHOP PATH SOLVER
# ======================================================

def compute_all_paths_from_source_multihop_no_earth_cuda(
    S_m: np.ndarray,
    sat_positions_m,
    rect_positions_m,
    visibility_mask,
    device: torch.device = None,
) -> list:
    """
    CUDA-accelerated version of your multihop Earth-safe path computation.

    - Uses GPU (if available) to check which segments intersect the Earth sphere
      for:
        S <-> sat
        sat <-> sat
        sat <-> rect
    - Then builds the graph on CPU and runs BFS (cheap).

    Parameters
    ----------
    S_m : np.ndarray, shape (3,)
        Source position in METERS (Earth center at origin).
    sat_positions_m : torch.Tensor or np.ndarray
        Shape (N_sats, T, 3), satellite positions in METERS.
    rect_positions_m : torch.Tensor or np.ndarray
        Shape (N_rect, T, 3), rectenna positions in METERS.
    visibility_mask : torch.Tensor or np.ndarray
        Shape (N_sats, N_rect, T), boolean:
          visibility_mask[i, r, t] == True if sat i can activate rect r at time t.
    device : torch.device or None
        If None, uses cuda if available, else cpu.

    Returns
    -------
    paths_matrix : list of lists
        Shape [N_rect][T].
        Each entry is either:
          - "Null" -> no feasible path that avoids Earth
          - [ [S_xyz], [sat_xyz], ..., [rect_xyz] ] in METERS
    """

    # ---- Device selection ----
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # ---- Convert everything to numpy / torch ----
    S_m = np.asarray(S_m, dtype=float)
    S_t = torch.from_numpy(S_m).to(device)  # (3,)

    if isinstance(sat_positions_m, torch.Tensor):
        sats_t_all = sat_positions_m.to(device).float()
    else:
        sats_t_all = torch.from_numpy(np.asarray(sat_positions_m, dtype=float)).to(device).float()

    if isinstance(rect_positions_m, torch.Tensor):
        rects_t_all = rect_positions_m.to(device).float()
    else:
        rects_t_all = torch.from_numpy(np.asarray(rect_positions_m, dtype=float)).to(device).float()

    if isinstance(visibility_mask, torch.Tensor):
        vis_all = visibility_mask.to(device).bool()
    else:
        vis_all = torch.from_numpy(np.asarray(visibility_mask, dtype=bool)).to(device)

    N_sats, T, _ = sats_t_all.shape
    N_rect = rects_t_all.shape[0]

    # Node indexing:
    #   0                 : source S
    #   1 .. N_sats       : satellites (sat i -> node 1 + i)
    #   N_sats+1 .. end   : rectennas (rect r -> node 1 + N_sats + r)
    def sat_node(i):
        return 1 + i

    def rect_node(r):
        return 1 + N_sats + r

    N_nodes = 1 + N_sats + N_rect

    # Initialize paths matrix as "Null"
    paths_matrix = [["Null" for _ in range(T)] for _ in range(N_rect)]

    radius_m = float(EARTH_RADIUS_M)

    # ---- Process each time step independently ----
    for t in tqdm(range(T), desc="Computing multi-hop Earth-safe paths (CUDA)"):
        # Slice positions & visibility for this time step (still on device)
        sats_t = sats_t_all[:, t, :]     # (N_sats, 3) on device
        rects_t = rects_t_all[:, t, :]   # (N_rect, 3) on device
        vis_t   = vis_all[:, :, t]       # (N_sats, N_rect) on device

        # --------------------------------------------------
        # 1) Build adjacency using GPU-based intersection checks
        # --------------------------------------------------

        # --- S <-> satellites: check intersection for S->sat_i ---
        S_rep = S_t.unsqueeze(0).expand(N_sats, 3)         # (N_sats, 3)
        intersects_S_sat = segments_intersects_sphere_torch(
            S_rep, sats_t, radius_m
        )                                                 # (N_sats,)
        s_sat_ok = (~intersects_S_sat)                    # allowed edges S <-> sat_i

        # --- sat <-> sat: all to all, check intersection ---
        pi = sats_t
        p1_ss = pi.unsqueeze(1).expand(N_sats, N_sats, 3)  # (N_sats, N_sats, 3)
        p2_ss = pi.unsqueeze(0).expand(N_sats, N_sats, 3)
        intersects_ss = segments_intersects_sphere_torch(
            p1_ss, p2_ss, radius_m
        )                                                 # (N_sats, N_sats)
        sat_sat_ok = (~intersects_ss)
        sat_sat_ok.fill_diagonal_(False)                  # no self-edges

        # --- sat <-> rect: check only where vis_t is True ---
        p1_sr = sats_t.unsqueeze(1).expand(N_sats, N_rect, 3)   # (N_sats, N_rect, 3)
        p2_sr = rects_t.unsqueeze(0).expand(N_sats, N_rect, 3)
        intersects_sr = segments_intersects_sphere_torch(
            p1_sr, p2_sr, radius_m
        )                                                       # (N_sats, N_rect)
        sat_rect_ok_geo = (~intersects_sr)
        sat_rect_ok = sat_rect_ok_geo & vis_t                   # only visible & Earth-safe

        # --------------------------------------------------
        # 2) Move adjacency info to CPU and build neighbor lists
        # --------------------------------------------------
        s_sat_ok_cpu    = s_sat_ok.detach().cpu().numpy()          # (N_sats,)
        sat_sat_ok_cpu  = sat_sat_ok.detach().cpu().numpy()        # (N_sats, N_sats)
        sat_rect_ok_cpu = sat_rect_ok.detach().cpu().numpy()       # (N_sats, N_rect)

        # Node positions (CPU) for BFS & path reconstruction
        node_pos = np.zeros((N_nodes, 3), dtype=float)
        node_pos[0, :] = S_m
        node_pos[1:1+N_sats, :] = sats_t.detach().cpu().numpy()
        node_pos[1+N_sats:, :] = rects_t.detach().cpu().numpy()

        # Build adjacency list on CPU
        neighbors = [[] for _ in range(N_nodes)]

        # --- S <-> satellites ---
        for i in range(N_sats):
            if s_sat_ok_cpu[i]:
                si = sat_node(i)
                neighbors[0].append(si)
                neighbors[si].append(0)

        # --- sat <-> sat ---
        for i in range(N_sats):
            ni = sat_node(i)
            row = sat_sat_ok_cpu[i]   # (N_sats,)
            for j in np.where(row)[0]:
                nj = sat_node(j)
                neighbors[ni].append(nj)
                # sat_sat_ok is symmetric, so we don't need to add reverse again explicitly

        # --- sat <-> rect ---
        for i in range(N_sats):
            ni = sat_node(i)
            row = sat_rect_ok_cpu[i]  # (N_rect,)
            for r in np.where(row)[0]:
                nr = rect_node(r)
                neighbors[ni].append(nr)
                neighbors[nr].append(ni)

        # --------------------------------------------------
        # 3) BFS from S (node 0) on CPU
        # --------------------------------------------------
        source_id = 0
        parent = [-1] * N_nodes
        parent[source_id] = source_id
        q = deque([source_id])

        while q:
            u = q.popleft()
            for v in neighbors[u]:
                if parent[v] == -1:
                    parent[v] = u
                    q.append(v)

        # --------------------------------------------------
        # 4) Reconstruct paths for each rectenna at time t
        # --------------------------------------------------
        for r in range(N_rect):
            target_id = rect_node(r)
            if parent[target_id] == -1:
                # unreachable → no feasible multihop path avoiding Earth
                continue

            # Reconstruct node id path from target back to source
            path_nodes = []
            cur = target_id
            while cur != source_id:
                path_nodes.append(cur)
                cur = parent[cur]
            path_nodes.append(source_id)
            path_nodes.reverse()  # now [S, ..., rect]

            # Convert to positions in METERS
            xyz_path = [node_pos[n].tolist() for n in path_nodes]
            paths_matrix[r][t] = xyz_path

    return paths_matrix


In [ ]:
# Example (assuming you've loaded your tensors already):

data = torch.load("/content/drive/MyDrive/space_robotics/rect_and_dat_pos.pt", map_location="cuda")
sat_positions_m = data["sat_positions"]      # (N_sats, T, 3)
rect_positions_m = data["rect_positions"]    # (N_rect, T, 3)
visibility_mask  = data["visibility_mask"]   # (N_sats, N_rect, T)

S_m = np.array([42371000, 0.0, 0.0], dtype=float)

paths_matrix = compute_all_paths_from_source_multihop_no_earth_cuda(
    S_m=S_m,
    sat_positions_m=sat_positions_m,
    rect_positions_m=rect_positions_m,
    visibility_mask=visibility_mask,
)

print("Example path for rectenna 0 at time 0:", paths_matrix[0][0])


Using device: cuda


Computing multi-hop Earth-safe paths (CUDA): 100%|██████████| 864/864 [02:58<00:00,  4.83it/s]


Example path for rectenna 0 at time 0: Null


In [ ]:
import os

save_path = "/content/drive/MyDrive/space_robotics/path_matrix_multihop_colab.pt"

# Make sure the directory exists
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# Save paths_matrix
torch.save({"paths_matrix": paths_matrix}, save_path)

In [ ]:
paths_matrix = compute_all_paths_from_source_multihop_no_earth_cuda(
    S_m=S_m,
    sat_positions_m=sat_positions_m,
    rect_positions_m=rect_positions_m,
    visibility_mask=visibility_mask,
)

print("Example path for rectenna 0 at time 0:", paths_matrix[0][0])


Using device: cuda


Computing multi-hop Earth-safe paths (CUDA): 100%|██████████| 864/864 [03:02<00:00,  4.73it/s]

Example path for rectenna 0 at time 0: Null
